In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
print("Installing bcftools...")
!sudo apt-get update -q
!sudo apt-get install bcftools -y -q

Installing bcftools...
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,887 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,6

In [5]:
import os
import subprocess
import shutil

In [4]:
INPUT_DIR = "/content/drive/MyDrive/FYP_DATA/DATA/raw_data/SG10K/"
ALIAS_FILE = "/content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/indian_alias.txt"
OUTPUT_DIR = "/content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered/"

In [6]:
# 1. Copy Alias file to local Colab disk ONCE
print("🚀 Setting up high-speed environment...")
if not os.path.exists("/content/indian_alias.txt"):
    shutil.copy(ALIAS_FILE, "/content/indian_alias.txt")
    # Ensure unix line endings just in case
    os.system("sed -i 's/\r$//' /content/indian_alias.txt")

LOCAL_ALIAS = "/content/indian_alias.txt"

🚀 Setting up high-speed environment...


In [7]:
chrom=2

In [8]:
drive_input = os.path.join(INPUT_DIR, f"chr{chrom}.vcf.gz")
drive_output = os.path.join(OUTPUT_DIR, f"chr{chrom}_sas_filtered.vcf.gz")

local_input = f"/content/chr{chrom}.vcf.gz"
local_output = f"/content/chr{chrom}_sas_filtered.vcf.gz"

In [9]:
# STEP A: Copy Input to Local SSD
print(f"  ⬇️  Downloading to local SSD...")
shutil.copy(drive_input, local_input)

  ⬇️  Downloading to local SSD...


'/content/chr2.vcf.gz'

In [10]:
# STEP B: Run bcftools locally
print(f"  ⚙️  Filtering & Recalculating AF...")
cmd = f"""
bcftools view -S "{LOCAL_ALIAS}" --force-samples -Ou "{local_input}" | \
bcftools plugin fill-tags -Oz -o "{local_output}" -- -t AF,AC,AN
"""
subprocess.run(cmd, shell=True, check=True)

  ⚙️  Filtering & Recalculating AF...


CompletedProcess(args='\nbcftools view -S "/content/indian_alias.txt" --force-samples -Ou "/content/chr2.vcf.gz" | bcftools plugin fill-tags -Oz -o "/content/chr2_sas_filtered.vcf.gz" -- -t AF,AC,AN\n', returncode=0)

In [11]:
# STEP C: Index Locally
print(f"  📇 Indexing...")
subprocess.run(f"bcftools index -t '{local_output}'", shell=True, check=True)

  📇 Indexing...


CompletedProcess(args="bcftools index -t '/content/chr2_sas_filtered.vcf.gz'", returncode=0)

In [12]:
# STEP D: Upload Result to Drive
print(f"  ⬆️  Uploading result to Drive...")
shutil.copy(local_output, drive_output)
shutil.copy(f"{local_output}.tbi", f"{drive_output}.tbi")

  ⬆️  Uploading result to Drive...


'/content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered/chr2_sas_filtered.vcf.gz.tbi'

In [13]:
# Check Final Size
size_mb = os.path.getsize(drive_output) / (1024 * 1024)
print(f"  ✅ Complete! Size: {size_mb:.2f} MB")

  ✅ Complete! Size: 561.66 MB


In [14]:
# Cleanup Local Files to prevent disk full
if os.path.exists(local_input): os.remove(local_input)
if os.path.exists(local_output): os.remove(local_output)
if os.path.exists(f"{local_output}.tbi"): os.remove(f"{local_output}.tbi")


# Bringing the files in colab's memory isn't effecting the processing time, so ignore this method.